# 1 — Getting started with seqcraft

seqcraft is **three things**, and one of them is pypulseq's.

| | What it is |
|---|---|
| `sc.LogicBlock` | a tree of pulseq events, each with a start time. Two attributes and one method. Anything may overlap anything. |
| `sc.compile` | turns that tree into legal pulseq blocks: finds boundaries, sums gradients that share an axis, validates against the amplifier. |
| `pp.Opts` | the scanner. Not wrapped, not subclassed — the same object you pass to `pp.make_trapezoid`. |

**This notebook uses no modules at all.** Every event below comes from raw pypulseq. That is
deliberate: it is the demonstration that the compile path stands alone, and it is also the honest
starting point, because seqcraft ships no concrete modules. `sc.Module` — the standard shape for a
*reusable* component you write yourself — appears at the end, once there is a reason for it.

In [1]:
import math

import numpy as np
import pypulseq as pp

import seqcraft as sc

sc.__version__

d:\MIniconda\envs\NUM\Lib\site-packages\sigpy\config.py:27: UserWarning: Importing cupy.cuda.cudnn failed. For more details, see the error stack below:
DLL load failed while importing cudnn: The specified module could not be found.
  warnings.warn(


'0.3.0'

## The scanner

A `pp.Opts`, built the ordinary way. Two kinds of number go into it and they come from different
places:

- **the amplifier** — `max_grad`, `max_slew`, `B0`. A spec sheet, or
  `sc.opts.from_scanner('Siemens Healthineers', 'MAGNETOM Prisma', ...)` which looks them up in
  [PulseqSystems](https://github.com/nimpulseq/PulseqSystems).
- **your installation** — the rasters, the dead times, the ringdown, the per-event sample limits.
  These are on no spec sheet and no vendor database has them.

**Set the dead times.** pypulseq defaults `rf_dead_time`, `rf_ringdown_time` and `adc_dead_time` to
**zero**, which is wrong on every real scanner. A sequence built on those compiles cleanly, passes
every check below, and is refused or silently mangled at the console.

In [2]:
opts = pp.Opts(
    # the amplifier
    max_grad=40, grad_unit='mT/m',
    max_slew=150, slew_unit='T/m/s',
    B0=3.0,
    # this installation -- not defaulted, because pypulseq's defaults are zero
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
)

print(f'{sc.convert(opts.max_grad, "Hz/m", "mT/m", gamma=opts.gamma):.0f} mT/m, '
      f'{sc.convert(opts.max_slew, "Hz/m/s", "T/m/s", gamma=opts.gamma):.0f} T/m/s, '
      f'{opts.rf_dead_time * 1e6:.0f} us RF dead time')

40 mT/m, 150 T/m/s, 100 us RF dead time


### A derated scanner is a second `Opts`

There are no named "regimes". A part that must be designed against weaker limits — an EPI train
derated for peripheral nerve stimulation — gets its own `Opts`.

Derive it with `sc.opts.derate`; do **not** write `pp.Opts(max_grad=...)` by hand. `Opts` fills
every argument you omit from the *process-global* default, so the hand-written version silently
returns your dead times to zero.

In [3]:
quiet = sc.opts.derate(opts, grad=0.8, slew=0.5)

print(f'derated slew: {quiet.max_slew / opts.max_slew:.0%} of full')
print(f'dead time carried across: {quiet.rf_dead_time == opts.rf_dead_time}')
print(f'the hand-written version loses it: '
      f'{pp.Opts(max_grad=opts.max_grad * 0.8).rf_dead_time} s')

derated slew: 50% of full
dead time carried across: True
the hand-written version loses it: 0 s


## A logic block

`LogicBlock.add(start, *items)` places pulseq events at a time **relative to the block**. That is
the whole data model: a list of `(start, item)` nodes, where an item is a pulseq event or another
block.

Here is one excitation — a slice-selective sinc, its slice-select gradient, and the rephaser that
undoes the dephasing the second half of the gradient caused.

In [4]:
rf, gz, gz_reph = pp.make_sinc_pulse(
    flip_angle=math.radians(15), duration=1e-3, slice_thickness=5e-3,
    apodization=0.5, time_bw_product=4,
    delay=opts.rf_dead_time, use='excitation',
    system=opts, return_gz=True,
)

excitation = sc.LogicBlock('excitation')
excitation.add(0.0, rf, gz)
excitation.add(pp.calc_duration(gz), gz_reph)

excitation

LogicBlock(excitation, 3 nodes, 1.80 ms)

In [5]:
print(excitation.describe())

excitation  1.80 ms
  +0.0 us  rf
  +0.0 us  trap z
  +1260.0 us  trap z


`duration` is **measured** from the nodes. Nothing declares it, so nothing can lie about it — which
is why a block is what you place the next thing by.

In [6]:
print(f'{excitation.duration * 1e3:.3f} ms')

1.800 ms


## A readout, and what TE actually means

A frequency-encoding lobe with an ADC on its flat top, plus the prephaser that walks k to the edge
of k-space first.

`time_to_echo` — where k = 0 falls *inside* the readout — is the one thing the block cannot tell
you. A block knows times, not meanings. So it is computed here, beside the gradient it describes.

In [7]:
FOV_MM, MATRIX = 250.0, 64
dk = 1e3 / FOV_MM                                    # 1/FOV, in 1/m

gx = pp.make_trapezoid('x', flat_area=MATRIX * dk, flat_time=3.2e-3, system=opts)
adc = pp.make_adc(num_samples=MATRIX, duration=3.2e-3, delay=gx.rise_time, system=opts)
gx_pre = pp.make_trapezoid('x', area=-gx.area / 2, duration=1e-3, system=opts)

# Sampling starts when the ramp ends, so k = 0 arrives half a flat top later.
time_to_echo = float(gx.rise_time) + 0.5 * float(gx.flat_time)

readout = sc.LogicBlock('readout').add(0.0, gx, adc)
print(f'readout {readout.duration * 1e3:.2f} ms, echo at {time_to_echo * 1e3:.2f} ms into it')

readout 3.24 ms, echo at 1.62 ms into it


## One TR

Three winders — the slice rephaser on z, the phase-encode blip on y, the readout prephaser on x —
are placed **at the same time on three axes**. Nothing coordinates them and nothing needs to: the
compiler puts them in one block and says nothing, because there is nothing wrong.

TE is then arithmetic, not a parameter anything stores:

```
t_readout = (excitation centre) + TE - (echo offset into the readout)
```

In [8]:
TE, TR = 8e-3, 20e-3
raster = sc.Raster(opts.block_duration_raster, 'block')

isodelay = float(rf.delay) + 0.5 * float(rf.shape_dur)     # a symmetric sinc refocuses mid-pulse
t_winders = float(pp.calc_duration(gz))
t_readout = raster.ceil(isodelay + TE - time_to_echo)
spoil = pp.make_trapezoid('z', area=4.0 / 5e-3, system=opts)

def one_tr(line: int) -> sc.LogicBlock:
    """A single spoiled gradient-echo TR, at phase-encode line `line`."""
    pe = pp.make_trapezoid('y', area=line * dk, duration=1e-3, system=opts)
    return (sc.LogicBlock(f'tr_{line}')
            .add(0.0, rf, gz)
            .add(t_winders, gz_reph, pe, gx_pre)       # three axes, one time
            .add(t_readout, gx, adc)
            .add(raster.ceil(t_readout + readout.duration), spoil))

print(one_tr(0).describe())

tr_0  10.99 ms
  +0.0 us  rf
  +0.0 us  trap z
  +1260.0 us  trap z
  +1260.0 us  trap y
  +1260.0 us  trap x
  +7010.0 us  trap x
  +7010.0 us  adc
  +10250.0 us  trap z


## Compile

`sc.compile(tree, opts)` takes the tree and the scanner, and nothing else. It never asks what
produced the tree.

In [9]:
seq = sc.LogicBlock('gre_2d')
for index, line in enumerate(range(-MATRIX // 2, MATRIX // 2)):
    seq.add(index * TR, one_tr(line))
    seq.add(index * TR + t_readout, pp.make_label('LIN', 'SET', line + MATRIX // 2))

out = sc.compile(seq, opts, name='gre_2d', definitions={'TE': TE, 'TR': TR})
out

CompiledSequence(gre_2d, 383 blocks, 1.271 s, 0 errors, 64 warnings)

In [10]:
report = out.check()
print('ok      :', report.ok)
print('errors  :', len(report.errors))
print('warnings:', len(report.warnings))
print()
print('one of them:', report.warnings[0].message)

ok      : True
errors  : 0
warnings: 64

one of them: slew norm on norm reaches 206.6 T/m/s, limit 150.0 T/m/s (138%); from gre_2d.tr_-32


Those warnings are the **vector norm** across simultaneous axes, and they are warnings on purpose.
Three winders ramping together reach up to `sqrt(3)` times the per-axis slew in vector magnitude,
which real amplifiers allow; making it an error would reject the ordinary overlap the whole design
exists to support. Per-axis violations are errors, and there are none.

### Provenance comes from the tags

Every compiled block records the tag path it came from. Nothing was registered and no bookkeeping
was written — the tags on the `LogicBlock`s are the whole mechanism. Blocks with an empty path are
the pure delays the compiler inserted between TRs; they came from no one.

In [11]:
for index in range(6):
    print(index, out.origin(index) or '(compiler-inserted delay)')

0 ('gre_2d', 'tr_-32')
1 ('gre_2d', 'tr_-32')
2 (compiler-inserted delay)
3 ('gre_2d',)
4 ('gre_2d', 'tr_-32')
5 (compiler-inserted delay)


### Check it against physics, not against itself

The only assertions worth making are ones an independent calculation gives.

In [12]:
k = out.kspace()
k_max = MATRIX / (2 * FOV_MM / 1e3)

window = k['t_adc'][:MATRIX]
echo_at = 0.5 * (window[MATRIX // 2 - 1] + window[MATRIX // 2]) - k['t_excitation'][0]

print(f'k_max          {np.abs(k["k_adc"][0]).max():7.1f} 1/m   (matrix/2FOV = {k_max:.1f})')
print(f'echo lands at  {echo_at * 1e3:7.3f} ms    (TE = {TE * 1e3:.3f})')

k_max            126.0 1/m   (matrix/2FOV = 128.0)
echo lands at    8.000 ms    (TE = 8.000)


## The three overlap rules

They are the whole of what the compiler decides, and each was chosen rather than fallen into.

| Overlap | What happens | Why |
|---|---|---|
| gradients on **different axes** | silent | it is the normal way to build a sequence; warning would teach people to ignore warnings |
| gradients on the **same axis** | warned, then summed | summing is almost always what was meant — but it is the one place a waveform changes, so it says so |
| two **RF** or two **ADC** | error | you cannot transmit twice at once, or transmit and receive at once |

In [13]:
gentle = {'area': 100.0, 'duration': 2e-3, 'rise_time': 200e-6, 'system': opts}

different_axes = sc.LogicBlock('ok').add(0.0, pp.make_trapezoid('x', **gentle),
                                         pp.make_trapezoid('y', **gentle))
same_axis = sc.LogicBlock('merged').add(0.0, pp.make_trapezoid('x', **gentle),
                                        pp.make_trapezoid('x', **gentle))

print('different axes:', sc.compile(different_axes, opts).report.warnings or 'nothing reported')
print('same axis     :', sc.compile(same_axis, opts).report.warnings[0].message)

different axes: nothing reported
same axis     : 2 gradients overlap on axis x over 0.0..2000.0 us and were summed: merged


Limits are checked **after** summing, which is the only place the truth is visible: two individually
legal gradients on one axis can sum to an illegal one, and no component can see that in isolation.

## Writing the file

`write()` drops the `.seq` and, beside it, a JSON sidecar recording versions, git state, the
definitions, the achieved timing, the file hash — and every field of the `Opts` it was built
against, so the scanner it was designed for is recoverable a year later.

In [14]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    result = out.write(Path(tmp) / 'gre_2d.seq')
    sidecar = json.loads(result.sidecar.read_text())

print(f'{result.n_blocks} blocks, {result.duration_s:.2f} s, sha256 {result.sha256[:12]}')
print('opts recorded:', sorted(sidecar['resolved']['opts'])[:6], '...')
print('rf_dead_time :', sidecar['resolved']['opts']['rf_dead_time'])

383 blocks, 1.27 s, sha256 adb7921d8959
opts recorded: ['B0', 'adc_dead_time', 'adc_raster_time', 'adc_samples_divisor', 'adc_samples_limit', 'block_duration_raster'] ...
rf_dead_time : 0.0001


## When a component is worth reusing: `sc.Module`

Everything above is a function of `opts` and a few numbers, which is fine for one sequence. A
component you will use in *several* sequences wants a shape, and `sc.Module` is that shape:
**parameters in, one `LogicBlock` out.**

Two rules make it work:

- **`__init__` designs, `build` assembles.** Waveforms are created once and stored on `self`; a call
  reads them and derives variants. Sixty-four phase-encode lines cost one design and sixty-four
  cheap calls.
- **calls are pure.** Never assign to `self.g.amplitude` — derive a modified event with
  `sc.events.derive`. The classic bug, `self.gx.amplitude = -self.gx.amplitude` in a readout loop,
  still compiles and makes TR 500 differ from TR 1.

Note there is no `duration` property. The block measures itself, so a build argument is free to
change the duration — which a declared duration had to forbid.

In [15]:
class PhaseEncode(sc.Module):
    """A phase-encode blip, designed once at its largest and scaled per line."""

    def __init__(self, *, opts, fov_mm, matrix, axis='y', tag=None):
        super().__init__(opts=opts, tag=tag)
        self.dk = 1e3 / float(fov_mm)
        self.g = pp.make_trapezoid(channel=axis, area=self.dk * matrix / 2, system=opts)

    def build(self, *, line: int = 0) -> sc.LogicBlock:
        scale = line * self.dk / float(self.g.area)
        return sc.LogicBlock().add(0.0, sc.events.derive(
            self.g,
            amplitude=float(self.g.amplitude) * scale,
            area=float(self.g.area) * scale,
            flat_area=float(self.g.flat_area) * scale,
        ))


pe = PhaseEncode(opts=opts, fov_mm=FOV_MM, matrix=MATRIX)
pe(line=17)

LogicBlock(PhaseEncode, 1 node, 0.30 ms)

`module(...)` is the interface; `build` is what you write. The base does exactly two things in
between: it rejects a `build` that returned the wrong type — with *your* class in the traceback —
and it names the block after the class if `build` left it untagged.

Designing at the **largest** area and scaling down is what keeps every line the same length, so the
caller's placement arithmetic does not depend on which line is being acquired.

In [16]:
print('tag       :', pe(line=17).tag)
print('same length:', pe(line=1).duration == pe(line=-32).duration)

sc.testing.assert_all(pe, line=17)      # raster, limits, purity, compiles alone
print('contract  : passes')

tag       : PhaseEncode
same length: True
contract  : passes


## The escape hatches

They exist so that "seqcraft cannot express this" is never the end of the conversation.

- **`sc.barrier()`** — force a block boundary where you need one.
- **`out.seq`** — the `pypulseq.Sequence`, yours to modify directly. seqcraft never hides it.
- **a plain function** — the compiler takes a `LogicBlock` and never asks what produced it, so a
  function returning one is a first-class component. `sc.Module` is a convention, not a gate.

## Where to go next

- [`docs/architecture.md`](../docs/architecture.md) — the layering, and what is deliberately absent.
- [`docs/compiler.md`](../docs/compiler.md) — how boundaries are chosen, for when you want to know
  why a particular block appeared.
- [`examples/_parked/`](_parked/) — two complete DTI acquisitions, kept as the specification for the
  module library that comes next. They do not run against this version.